In [123]:
import numpy as np
import scipy.linalg as la
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy.optimize import minimize
from scipy.linalg import norm
from scipy.linalg import expm, sinm, cosm
import random
import cvxpy as cp
import itertools
import random
from scipy.linalg import eigh


sx = np.array([[0, 1], [1, 0]])  
sz = np.array([[1, 0], [0, -1]])  
sy = np.array([[0, -1j], [1j, 0]])  
identity = np.eye(2)  
I = np.eye(2)
s_plus = np.array([[0, 1], [0, 0]])
s_minus = np.array([[0, 0], [1, 0]])

def kron_n(*matrices):
    result = np.array([[1.0]])  
    for matrix in matrices:
        result = np.kron(result, matrix)
    return result

In [124]:
def annni_hamiltonian(
    L,
    J1=1.0,      # NN X-X coupling
    J2=0.0,      # NNN X-X coupling
    B=0.0,       # Longitudinal field (sigma^z)
    G=0.0,       # Transverse field (sigma^x)
    Jy=0.0,      # Y-Y coupling
    Delta=0.0,   # Z-Z coupling
    periodic=True
):
    """
    General Hamiltonian:
    
    H = -J1 * Σ σ^x_i σ^x_{i+1}
        -J2 * Σ σ^x_i σ^x_{i+2}
        -B  * Σ σ^z_i
        -G  * Σ σ^x_i
        -Jy * Σ σ^y_i σ^y_{i+1}
        -Δ  * Σ σ^z_i σ^z_{i+1}
    
    Includes:
    • ANNNI: J1 != 0, J2 != 0, G=Jy=Δ=0
    • XXZ + transverse field: J2=0, Jy=J1, Δ free
    • Longitudinal+Transverse Ising: J2=Jy=Δ=0
    """

    dim = 2**L
    H = np.zeros((dim, dim), dtype=complex)

    # --- Helper Pauli strings ---
    def op_on_sites(op_i, op_j=None, j=None):
        ops = [identity]*L
        ops[op_i[0]] = op_i[1]
        if op_j is not None and j is not None:
            ops[j] = op_j
        return kron_n(*ops)

    # ========== 1) Nearest-neighbour X-X ==========
    for i in range(L if periodic else L-1):
        j = (i+1) % L
        H += -J1 * kron_n(*([sx if k==i else sx if k==j else identity for k in range(L)]))

    # ========== 2) Next-nearest-neighbour X-X ==========
    if L >= 3:
        for i in range(L if periodic else L-2):
            j = i+2
            if not periodic and j >= L:
                continue
            j = j % L
            H += -J2 * kron_n(*([sx if k==i else sx if k==j else identity for k in range(L)]))

    # ========== 3) Y-Y coupling ==========
    for i in range(L if periodic else L-1):
        j = (i+1) % L
        H += -Jy * kron_n(*([sy if k==i else sy if k==j else identity for k in range(L)]))

    # ========== 4) Z-Z coupling ==========
    for i in range(L if periodic else L-1):
        j = (i+1) % L
        H += -Delta * kron_n(*([sz if k==i else sz if k==j else identity for k in range(L)]))

    # ========== 5) Longitudinal field B Σ Z_i ==========
    if B != 0:
        for i in range(L):
            H += -B * kron_n(*([sz if k==i else identity for k in range(L)]))

    # ========== 6) Transverse field G Σ X_i ==========
    if G != 0:
        for i in range(L):
            H += -G * kron_n(*([sx if k==i else identity for k in range(L)]))

    return H


In [125]:
def pauli_operator(pauli_string):
    # print pauli_string
    op_list = [identity] * len(pauli_string)  # initialize the operator list with identity matrices

    for i, char in enumerate(pauli_string):
        if char == 'X':
            op_list[i] = sx
        elif char == 'Y':
            op_list[i] = sy
        elif char == 'Z':
            op_list[i] = sz
        elif char == 'I':
            op_list[i] = identity

    return kron_n(*op_list)

def pauli_expectation(rho, pauli_string, tol=1e-8):
    P = pauli_operator(pauli_string)
    exp_value = np.trace(rho @ P)
    
    if np.abs(exp_value.imag) > tol:
        print(f"[Warning] Large imaginary part detected: {exp_value.imag} for Pauli string {pauli_string}")
    
    return exp_value.real

def random_pauli_strings(L, Ns, seed=None):
    
    pauli_strings = []  

    if seed is not None:
        #np.random.seed(seed)
        random.seed(seed)

    for _ in range(Ns):
        
        # pick randomly L elemnents from 'I', 'X', 'Y', 'Z' and make a string to append to pauli_strings
        string = ''.join(random.choices(['I', 'X', 'Y', 'Z'], k=L))
        pauli_strings.append(string)

    return pauli_strings

def unique_random_pauli_strings(L, Ns, seed=None):
    if seed is not None:
        random.seed(seed)

    paulis = ['I', 'X', 'Y', 'Z']
    all_strings = [''.join(p) for p in itertools.product(paulis, repeat=L)]

    if Ns > len(all_strings):
        raise ValueError(f"Requested {Ns} strings, but only {len(all_strings)} unique strings available.")

    return random.sample(all_strings, Ns)

def pauli_expectations(rho, strings):
   
    values = []

    for s in strings:
        expectation = pauli_expectation(rho, s)
        values.append(expectation)

    return np.array(values)

In [126]:
import itertools
import random

def unique_random_pauli_strings_fewbodies(L, Ns, seed=None):
    """
    Generate up to Ns unique Pauli strings prioritizing few-body terms:
    1-body first, then 2-body, then 3-body, etc.
    Randomized *within* each body order, but ordered by increasing body size.

    Parameters:
        L (int): Number of qubits.
        Ns (int): Number of unique Pauli strings to return.
        seed (int, optional): Random seed for reproducibility.

    Returns:
        list[str]: List of unique Pauli strings (length L).
    """
    if seed is not None:
        random.seed(seed)

    paulis = ['I', 'X', 'Y', 'Z']
    ordered_strings = []

    # Loop over increasing body order
    for n in range(1, L + 1):
        n_body_strings = []
        for positions in itertools.combinations(range(L), n):
            for ops in itertools.product(['X', 'Y', 'Z'], repeat=n):
                s = ['I'] * L
                for pos, op in zip(positions, ops):
                    s[pos] = op
                n_body_strings.append(''.join(s))
        # Shuffle within each n-body group
        random.shuffle(n_body_strings)
        ordered_strings.extend(n_body_strings)

        # Stop once we've reached Ns
        if len(ordered_strings) >= Ns:
            break

    if Ns > len(ordered_strings):
        raise ValueError(f"Requested {Ns} unique Pauli strings, but only {len(ordered_strings)} available for L={L}.")

    return ordered_strings[:Ns]


In [127]:
def optimal_unitary(rho, H):
    # Diagonalize rho: descending eigenvalues
    r_evals, r_evecs = np.linalg.eigh(rho)
    r_sort_idx = np.argsort(-r_evals)
    r_evecs = r_evecs[:, r_sort_idx]

    # Diagonalize H: ascending eigenvalues
    H_evals, H_evecs = np.linalg.eigh(H)
    H_sort_idx = np.argsort(H_evals)
    H_evecs = H_evecs[:, H_sort_idx]

    # Build the unitary: U = sum_j |E_j><r_j|
    U = H_evecs @ r_evecs.conj().T
    return U

In [128]:
import cvxpy as cp

def sdp_minimization_relaxed(L, pauli_strings, pauli_values, U, H, epsilon=1e-8):
    # define the variable for the density matrix rho (LxL matrix)
    rho = cp.Variable((2**L, 2**L), hermitian=True)
    
    # Objective function: E(rho, H) = Tr[rho H] - Tr[U† rho U H]
    objective = cp.Minimize(cp.real(cp.trace((H - U.conj().T @ H @ U) @ rho)))

    # Constraints
    constraints = [
        # Tr(rho) = 1 constraint (normalization)
        cp.trace(rho) == 1,
        # rho should be positive semidefinite
        rho >> 0, 
    ]

    # Add the constraints Tr(Oi * rho) = oi for each Pauli string
    for i, pauli_string in enumerate(pauli_strings):
        Oi = pauli_operator(pauli_string)
        oi = cp.real(pauli_values[i])
        constraints.append(cp.real(cp.trace(rho @ Oi)) >= oi - epsilon)
        constraints.append(cp.real(cp.trace(rho @ Oi)) <= oi + epsilon)

    try:
        problem = cp.Problem(objective, constraints)
        problem.solve(solver=cp.SCS, verbose=False)

        if problem.status not in ["optimal", "optimal_inaccurate"]:
            print(f"[SDP Warning] Problem status: {problem.status}")
            return np.inf, None

        return problem.value, rho.value
    
    except Exception as e:
        print(f"[SDP Error] Solver failed: {e}")
        return np.inf, None


In [129]:
def compute_epsilon(Ns, n_shots=10000, delta=0.003):
    K = Ns
    return np.sqrt(2 * np.log(2 * K / delta) / n_shots)

In [130]:
def pauli_expectations_fake_shots(rho, pauli_list, n_shots, seed=None):
    """
    Simulates Pauli expectation values with finite statistics ("fake shots").
    Vectorized for speed.
    """
    if seed is not None:
        rng = np.random.default_rng(seed)
    else:
        rng = np.random.default_rng()

    expectations = []

    for P in pauli_list:
        operator = pauli_operator(P)
        exp_val = np.real(np.trace(rho @ operator))
        
        prob_plus = np.clip((1 + exp_val) / 2, 0, 1)

        # Fast binary sampling using uniform [0,1)
        rand_vals = rng.random(n_shots)
        outcomes = np.where(rand_vals < prob_plus, 1, -1)
        estimated_val = np.mean(outcomes)

        assert np.abs(estimated_val) <= 1.0 + 1e-6, f"Out-of-range value: {estimated_val}"
        expectations.append(estimated_val)

    return np.array(expectations)

In [131]:
def sdp_most_mixed_state_relaxed(L, pauli_strings, pauli_values, epsilon, solver=cp.SCS):
    """
    Return the most mixed (minimum Tr(rho^2)) feasible rho 
    given Pauli constraints and normalization.
    """
    rho = cp.Variable((2**L, 2**L), hermitian=True)
    purity = cp.sum_squares(cp.abs(rho))
    objective = cp.Minimize(cp.real(purity))

    constraints = [cp.trace(rho) == 1, rho >> 0]
    for i, pauli_string in enumerate(pauli_strings):
        Oi = pauli_operator(pauli_string)
        oi = cp.real(pauli_values[i])
        constraints.append(cp.real(cp.trace(rho @ Oi)) >= oi - epsilon)
        constraints.append(cp.real(cp.trace(rho @ Oi)) <= oi + epsilon)

    try:
        problem = cp.Problem(objective, constraints)
        problem.solve(solver=solver, verbose=False)
        if problem.status not in ["optimal", "optimal_inaccurate"]:
            print(f"[SDP Warning] status: {problem.status}")
            return None
        return rho.value
    except Exception as e:
        print(f"[SDP Error] {e}")
        return None

In [ ]:
# ============================================================
# ---- run one realization
# ============================================================
def run_single_realization(Ns, realization, L, rho, U_init, H, n_shots):
    seed = 42 + realization
    epsilon_val = compute_epsilon(Ns, n_shots)
    random_pauli = unique_random_pauli_strings_fewbodies(L, Ns, seed)

    # Step 1: Fake-shot Pauli measurements
    values = pauli_expectations_fake_shots(rho, random_pauli, n_shots, seed=seed)

    # Step 2: First relaxed SDP
    rho_feas = sdp_most_mixed_state_relaxed(L, random_pauli, values, epsilon_val)
    if rho_feas is None or rho_feas.ndim < 2:
        return None

    # Step 3: Optimal unitary from feasible rho
    U_optimal = optimal_unitary(rho_feas, H)

    # Step 4: Second relaxed SDP
    min_value_2, _ = sdp_minimization_relaxed(L, random_pauli, values, U_optimal, H, epsilon_val)
    if not np.isfinite(min_value_2):
        return None

    return min_value_2

In [133]:
L = 5

In [134]:
dim = 2**L
psi_uniform = np.ones(dim, dtype=complex) / np.sqrt(dim)
rho_uniform = np.outer(psi_uniform, psi_uniform.conj())

In [135]:
H_annni = annni_hamiltonian(L, J1=1.0, J2=-1.0, B=0.5, G=0.0, Jy=0.0, Delta=0.0)

In [136]:
# Assume H is your Hamiltonian matrix (shape 2^L x 2^L)
eigenvalues, eigenvectors = eigh(H_annni)

# Get eigenvectors corresponding to min and max eigenvalues
psi_min = eigenvectors[:, 0]                # Least excited
psi_max = eigenvectors[:, -1]               # Most excited

# Construct normalized superposition
psi_super = (psi_min + psi_max) / np.linalg.norm(psi_min + psi_max)

# Create density matrix
rho_super = np.outer(psi_super, psi_super.conj())

In [70]:
# Cell A: run parallel fake-shot reconstructions and save realizations per Ns
from joblib import Parallel, delayed
from tqdm import tqdm
import numpy as np
import os

# PARAMETERS (edit if needed)
Ns_list = list(range(1, 1000, 30))   
num_realizations = 20               
n_jobs = -1                         # -1 uses all CPUs
n_shots = 100000                     # shots PER pauli 


# output directory
outdir = "fake_shots_results"
os.makedirs(outdir, exist_ok=True)

# wrapper for Parallel (re-uses your run_single_realization)
def run_and_save_for_Ns(Ns):
    results = Parallel(n_jobs=n_jobs)(
        delayed(run_single_realization)(Ns, realization, L, rho_super, np.eye(2**L), H_annni, n_shots)
        for realization in range(num_realizations)
    )
    # filter out None
    realization_values = np.array([v for v in results if v is not None])
    if realization_values.size == 0:
        print(f"[Warning] No valid realizations for Ns={Ns}. File not saved.")
        return None

    filename = os.path.join(outdir, f"realization_fake_shots_{Ns}paulis_real{num_realizations}_qubits{L}_ANNNI.txt")
    np.savetxt(filename, realization_values)
    print(f"Saved {filename}  (N={Ns}, n_valid={len(realization_values)})")
    return filename

# Run over all Ns (progress bar)
for Ns in tqdm(Ns_list, desc="Running fake-shot sims"):
    run_and_save_for_Ns(Ns)

print("All done. Files saved in:", outdir)


Running fake-shot sims:   3%|▎         | 1/34 [00:00<00:28,  1.15it/s]

Saved fake_shots_results/realization_fake_shots_1paulis_real20_qubits5_ANNNI.txt  (N=1, n_valid=20)


Running fake-shot sims:   6%|▌         | 2/34 [00:02<00:34,  1.07s/it]

Saved fake_shots_results/realization_fake_shots_31paulis_real20_qubits5_ANNNI.txt  (N=31, n_valid=20)


Running fake-shot sims:   9%|▉         | 3/34 [00:04<00:45,  1.48s/it]

...

Saved fake_shots_results/realization_fake_shots_931paulis_real20_qubits5_ANNNI.txt  (N=931, n_valid=20)


Running fake-shot sims:  97%|█████████▋| 33/34 [08:30<00:27, 27.39s/it]

Saved fake_shots_results/realization_fake_shots_961paulis_real20_qubits5_ANNNI.txt  (N=961, n_valid=20)


Running fake-shot sims: 100%|██████████| 34/34 [08:58<00:00, 15.85s/it]

Saved fake_shots_results/realization_fake_shots_991paulis_real20_qubits5_ANNNI.txt  (N=991, n_valid=20)
All done. Files saved in: fake_shots_results


In [71]:
# Cell B: load saved realization files and compute median + IQR
import numpy as np
import os

def compute_quartile_stats_from_dir(prefix, Ns_list, num_realizations, L, outdir="fake_shots_results"):
    medians, lower, upper = [], [], []
    loaded_Ns = []
    for Ns in Ns_list:
        fname = os.path.join(outdir, f"{prefix}_{Ns}paulis_real{num_realizations}_qubits{L}_ANNNI.txt")
        if not os.path.exists(fname):
            print(f"[Warning] Missing file {fname} — skipping this Ns")
            continue
        data = np.loadtxt(fname)
        if data.size == 0:
            print(f"[Warning] File {fname} empty — skipping")
            continue
        # ensure 1D
        data = np.atleast_1d(data)
        med = np.median(data)
        q25, q75 = np.percentile(data, [25, 75])
        medians.append(med)
        lower.append(med - q25)
        upper.append(q75 - med)
        loaded_Ns.append(Ns)
    return np.array(loaded_Ns), np.array(medians), np.array(lower), np.array(upper)

# Use it:
prefix = "realization_fake_shots"
loaded_Ns, median_fs, lower_err_fs, upper_err_fs = compute_quartile_stats_from_dir(
    prefix, Ns_list, num_realizations, L, outdir=outdir
)

print("Loaded Ns (successful):", loaded_Ns)


Loaded Ns (successful): [  1  31  61  91 121 151 181 211 241 271 301 331 361 391 421 451 481 511
 541 571 601 631 661 691 721 751 781 811 841 871 901 931 961 991]


In [137]:
# True ergotropy for rho_super
p_super, _ = np.linalg.eigh(rho_super)
e_super, _ = np.linalg.eigh(H_annni)

# Sort probabilities (desc) and energies (asc)
p_sorted_super = np.sort(p_super)[::-1]
e_sorted_super = np.sort(e_super)

# Compute ergotropy
energy_init_super = np.trace(rho_super @ H_annni)
energy_min_super = np.sum(p_sorted_super * e_sorted_super)
ergotropy_super = energy_init_super - energy_min_super

print(f"True ergotropy (superposition state): {ergotropy_super:.6f}")

True ergotropy (superposition state): 5.259756+0.000000j


In [138]:
# Cell C: compute true ergotropy for rho_super and H_annni (prints energies too)
import numpy as np

def true_ergotropy(rho, H):
    # eigenvalues of rho and H
    p, _ = np.linalg.eigh(rho)
    e, _ = np.linalg.eigh(H)
    p_sorted = np.sort(p)[::-1]    # descending
    e_sorted = np.sort(e)          # ascending
    energy_init = np.real(np.trace(rho @ H))
    energy_min = np.sum(p_sorted * e_sorted)
    return energy_init - energy_min, energy_init, energy_min

ergotropy_super, energy_init_super, energy_min_super = true_ergotropy(rho_super, H_annni)
print(f"True ergotropy (superposition): {ergotropy_super:.8e}")
print(f"Energy init = {energy_init_super:.8e}, Energy min = {energy_min_super:.8e}")


True ergotropy (superposition): 5.25975625e+00
Energy init = 1.77635684e-15, Energy min = -5.25975625e+00


In [74]:
# Cell D: plotting (no latex), shaded IQR, median markers, red true-line
import numpy as np
import matplotlib.pyplot as plt
import pylab

# plotting params (match your no-LaTeX style)
fig_width_pt = 246.0
inches_per_pt = 1.0 / 72.27
fig_width = fig_width_pt * inches_per_pt
fig_size = [fig_width, 0.75 * fig_width]

params = {
    'backend': 'ps',
    'axes.labelsize': 10,
    'font.size': 9,
    'legend.fontsize': 9,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'text.usetex': False,
    'figure.figsize': fig_size
}
pylab.rcParams.update(params)

# Basic checks
if loaded_Ns.size == 0:
    raise RuntimeError("No data loaded. Run Cell A and Cell B first and ensure files exist.")

# plot arrays from Cell B: loaded_Ns, median_fs, lower_err_fs, upper_err_fs
x = loaded_Ns
y = median_fs
y1 = y - lower_err_fs
y2 = y + upper_err_fs

fig, ax = plt.subplots()

# padding so first point visible
x_min = x.min()
x_max = x.max()
padding = 0.05 * (x_max - x_min)
plt.xlim([x_min - padding, x_max + padding])

# shaded IQR
color_band = "#9884b9"   # violet tone consistent with before
ax.fill_between(x, y1, y2, color=color_band, alpha=0.25)

# median line + markers
ax.plot(x, y, 'o-', color=color_band, markersize=5, label="Reconstructed lower bound (median)")

# true ergotropy line
ax.axhline(y=ergotropy_super, color="#EE4B2B", linestyle='--', linewidth=1.4, label="True ergotropy")

# zero line
ax.axhline(y=0, color='gray', linestyle='--', linewidth=0.8)

# labels
ax.set_xlabel("Number of Pauli constraints")
ax.set_ylabel(r"$\mathcal{E}(\rho)$")   # you said earlier you wanted this LaTeX-y label; matplotlib will render plain if no latex

# grid & ticks
ax.grid(True, which='major', linestyle='--', linewidth=0.5)
ax.grid(True, which='minor', linestyle=':', linewidth=0.2)
for lab in (ax.get_xticklabels() + ax.get_yticklabels()):
    lab.set_fontsize(9)

ax.legend(fontsize=9)
plt.tight_layout()

# Save
fname = f"ergotropy_vs_Ns_fake_shots_super_ANNNI_L{L}_shots{n_shots}_reals{num_realizations}.pdf"
plt.savefig(fname, dpi=300, bbox_inches='tight')
print(f"Saved figure: {fname}")

# Show
plt.show()


Saved figure: ergotropy_vs_Ns_fake_shots_super_ANNNI_L5_shots100000_reals20.pdf


/var/folders/bl/2qzs5zlx0_b0dtwkqzfj3fznfc_zz4/T/ipykernel_2534/3902940151.py:74: UserWarning: FigureCanvasPS is non-interactive, and thus cannot be shown
  plt.show()


In [139]:
# Different shot numbers to test
shot_list = [10000, 50000, 200000]

# Colors for the curves (lighter → fewer shots)
import matplotlib.cm as cm
colors = cm.Blues(np.linspace(0.4, 0.9, len(shot_list)))

In [140]:
Ns_list = list(range(1, 1000, 40))
num_realizations = 20
n_jobs = -1     # use all CPUs

results_by_shots = {}   # dictionary: n_shots → (median, low, high)

for n_shots in shot_list:
    print(f"\n=== Running simulations for n_shots = {n_shots} ===")

    # Save all realization results for each Ns
    all_medians = []
    all_lowers = []
    all_uppers = []

    for Ns in tqdm(Ns_list):
        # Parallel evaluations for current Ns and n_shots
        vals = Parallel(n_jobs=n_jobs)(
            delayed(run_single_realization)(Ns, realization, L, rho_super,
                                            np.eye(2**L), H_annni, n_shots)
            for realization in range(num_realizations)
        )

        vals = np.array([v for v in vals if v is not None])

        # Compute statistics
        q25, q75 = np.percentile(vals, [25, 75])
        median = np.median(vals)

        all_medians.append(median)
        all_lowers.append(median - q25)
        all_uppers.append(q75 - median)

    # Store everything
    results_by_shots[n_shots] = (
        np.array(all_medians),
        np.array(all_lowers),
        np.array(all_uppers),
    )

print("All simulations completed.")



=== Running simulations for n_shots = 10000 ===


100%|██████████| 25/25 [06:11<00:00, 14.85s/it]



=== Running simulations for n_shots = 50000 ===


100%|██████████| 25/25 [07:28<00:00, 17.92s/it]



=== Running simulations for n_shots = 200000 ===


100%|██████████| 25/25 [09:56<00:00, 23.85s/it]

All simulations completed.


In [141]:
def true_ergotropy_scalar(rho, H):
    """
    Compute the true ergotropy of a state rho with respect to Hamiltonian H.

    Ergotropy = Tr[rho H] - sum_j p_sorted_desc[j] * E_sorted_asc[j]
    """
    # Diagonalize rho
    p, _ = np.linalg.eigh(rho)
    p_sorted = np.sort(p)[::-1]     # probabilities sorted descending

    # Diagonalize H
    E, _ = np.linalg.eigh(H)
    E_sorted = np.sort(E)           # energies sorted ascending

    # Initial energy
    E_init = np.real(np.trace(rho @ H))

    # Passive state's minimum energy
    E_min = np.sum(p_sorted * E_sorted)

    # Ergotropy
    return E_init - E_min


In [89]:
#########   imports (same style)
import matplotlib.pyplot as plt
import pylab

####--- Plot style (NO LATEX) ----####
fig_width_pt = 246.0
inches_per_pt = 1.0 / 72.27
fig_width = fig_width_pt * inches_per_pt
fig_size = [fig_width, 0.75 * fig_width]

params = {
    'backend': 'ps',
    'axes.labelsize': 10,
    'font.size': 9,
    'legend.fontsize': 9,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'text.usetex': False,
    'figure.figsize': fig_size
}
pylab.rcParams.update(params)

# safety check
if len(results_by_shots) == 0:
    raise RuntimeError("results_by_shots is empty. Run computation first.")

# ----- fixed colors for each shot number -----
shot_color_map = {
    min(results_by_shots.keys()): "#4C72B0",   # smallest shots
    10000: "#3C5B8C",
    50000: "#2D4469",
    max(results_by_shots.keys()): "#1E2D46"    # largest shots
}

# load Ns list
x = loaded_Ns
x_min, x_max = x.min(), x.max()
padding = 0.05 * (x_max - x_min)

# true ergotropy (scalar)
erg_true = true_ergotropy_scalar(rho_super, H_annni)

########--- PLOT ---########
fig, ax = plt.subplots()
plt.xlim([x_min - padding, x_max + padding])

for n_shots, (med, low, up) in results_by_shots.items():

    # choose color (fall back to a default if missing)
    color = shot_color_map.get(n_shots, "#4C72B0")

    # shaded IQR
    ax.fill_between(
        x,
        med - low,
        med + up,
        color=color,
        alpha=0.25
    )

    # median line
    ax.plot(
        x, med,
        color=color,
        marker='o',
        markersize=4,
        linewidth=1.5,
        label=f"{n_shots:,} shots"
    )

    # error bars (IQR)
    ax.errorbar(
        x, med,
        yerr=[low, up],
        fmt='none',
        ecolor=color,
        elinewidth=1.0,
        capsize=2
    )

# true ergotropy
ax.axhline(
    y=erg_true,
    color="#EE4B2B",
    linestyle='--',
    linewidth=1.4,
    label="True ergotropy"
)

# zero line
ax.axhline(
    y=0,
    color='gray',
    linestyle='--',
    linewidth=0.8
)

# labels
ax.set_xlabel("Number of Pauli constraints")
ax.set_ylabel(r"$\mathcal{E}(\rho)$")

# grid
ax.grid(True, which='major', linestyle='--', linewidth=0.5)
ax.grid(True, which='minor', linestyle=':', linewidth=0.2)

# ticks
for lab in (ax.get_xticklabels() + ax.get_yticklabels()):
    lab.set_fontsize(9)

ax.legend(fontsize=8)
plt.tight_layout()

# save
fname = f"ergotropy_vs_Ns_superposition_multishots_ANNNI_L{L}_colours.pdf"
plt.savefig(fname, dpi=300, bbox_inches='tight')
print(f"Saved figure: {fname}")

plt.show()


Saved figure: ergotropy_vs_Ns_superposition_multishots_ANNNI_L5_colours.pdf


/var/folders/bl/2qzs5zlx0_b0dtwkqzfj3fznfc_zz4/T/ipykernel_2534/3476958764.py:118: UserWarning: FigureCanvasPS is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
#########   imports (same style)
import matplotlib.pyplot as plt
import pylab

####--- Plot style (NO LATEX) ----####
fig_width_pt = 246.0
inches_per_pt = 1.0 / 72.27
fig_width = fig_width_pt * inches_per_pt
fig_size = [fig_width, 0.75 * fig_width]

params = {
    'backend': 'ps',
    'axes.labelsize': 10,
    'font.size': 9,
    'legend.fontsize': 9,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'text.usetex': False,
    'figure.figsize': fig_size
}
pylab.rcParams.update(params)

# Safety check
if len(results_by_shots) == 0:
    raise RuntimeError("results_by_shots is empty. Run computation first.")

# ----- fixed colors for each shot number -----
shot_color_map = {
    min(results_by_shots.keys()): "#4C72B0",   # smallest shots
    10000: "#3C5B8C",
    50000: "#2D4469",
    max(results_by_shots.keys()): "#1E2D46"    # largest shots
}

x = loaded_Ns
x_min, x_max = x.min(), x.max()
padding = 0.05 * (x_max - x_min)

erg_true = true_ergotropy_scalar(rho_super, H_annni)

########--- PLOT ---########
fig, ax = plt.subplots()
plt.xlim([x_min - padding, x_max + padding])

for n_shots, (med, low, up) in results_by_shots.items():

    color = shot_color_map.get(n_shots, "#4C72B0")

    # ---- IQR shaded band ----
    ax.fill_between(
        x,
        med - low,
        med + up,
        color=color,
        alpha=0.25
    )

    # ---- median curve ----
    ax.plot(
        x, med,
        color=color,
        marker='o',
        markersize=5,
        linewidth=1.5,
        label=f"{n_shots:,} shots"
    )

# ---- true ergotropy ----
ax.axhline(
    y=erg_true,
    color="#EE4B2B",
    linestyle='--',
    linewidth=1.4,
    label="True ergotropy"
)

# zero line
ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)

ax.set_xlabel("Number of Pauli constraints")
ax.set_ylabel(r"$\mathcal{E}(\rho)$")

# grid
ax.grid(True, which='major', linestyle='--', linewidth=0.5)
ax.grid(True, which='minor', linestyle=':', linewidth=0.2)

# ticks but small
for lab in (ax.get_xticklabels() + ax.get_yticklabels()):
    lab.set_fontsize(9)

ax.legend(fontsize=8)
plt.tight_layout()

fname = f"ergotropy_vs_Ns_superposition_multishots_ANNNI_L{L}.pdf"
plt.savefig(fname, dpi=300, bbox_inches='tight')
print(f"Saved figure: {fname}")

plt.show()


Saved figure: ergotropy_vs_Ns_superposition_multishots_ANNNI_L5_noerrorbar.pdf


/var/folders/bl/2qzs5zlx0_b0dtwkqzfj3fznfc_zz4/T/ipykernel_2534/3350149011.py:98: UserWarning: FigureCanvasPS is non-interactive, and thus cannot be shown
  plt.show()


Trying to modify in the definitive form:

In [145]:
#########   imports (same style)
import numpy as np
import matplotlib.pyplot as plt
import pylab

####--- Plot style (NO LATEX) ----####
fig_width_pt = 246.0
inches_per_pt = 1.0 / 72.27
fig_width = fig_width_pt * inches_per_pt
fig_size = [fig_width, 0.75 * fig_width]

params = {
    'backend': 'ps',
    'axes.labelsize': 10,
    'font.size': 9,
    'legend.fontsize': 9,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'text.usetex': False,
    'figure.figsize': fig_size
}
pylab.rcParams.update(params)

# Safety check
if len(results_by_shots) == 0:
    raise RuntimeError("results_by_shots is empty. Run computation first.")

# ----- fixed colors for each shot number -----
shot_color_map = {
    10000: "#4C72B0",   # fewest shots  (lightest blue)
    50000: "#3C5B8C",
    200000: "#1E2D46"   # most shots    (darkest blue)
}

# x-axis values
#x = loaded_Ns
x = np.array(Ns_list)   # this has 25 elements
x_min, x_max = x.min(), x.max()
padding = 0.05 * (x_max - x_min)

# true ergotropy
erg_true = true_ergotropy_scalar(rho_super, H_annni)

########--- PLOT ---########
fig, ax = plt.subplots()
plt.xlim([x_min - padding, x_max + padding])

# line styles consistent with negative-temperature plot
plt.rc('lines', linewidth=1.2)
plt.rc('axes', linewidth=1.1)

for n_shots, (med, low, up) in results_by_shots.items():

    if n_shots not in shot_color_map:
        continue  # skip unexpected keys

    color = shot_color_map[n_shots]

    # ---- CLIP NEGATIVE VALUES ----
    med_clipped = np.maximum(med, 0)
    low_band = np.maximum(med - low, 0)
    up_band  = np.maximum(med + up, 0)

    # ---- IQR shaded band ----
    ax.fill_between(
        x,
        low_band,
        up_band,
        color=color,
        alpha=0.25
    )

    # ---- median line ----
    ax.plot(
        x, med_clipped,
        color=color,
        marker='o',
        markersize=3,
        linewidth=1.2,
        label=f"{n_shots:,} shots"
    )

# ---- true ergotropy ----
ax.axhline(
    y=erg_true,
    color="#EE4B2B",
    linestyle='--',
    linewidth=1.2,
    #label="True ergotropy"
)

# ---- zero line ----
ax.axhline(0, color='gray', linestyle='-', linewidth=0.6)

# ---- Labels ----
plt.xlabel(r"$|\mathcal{I}|$")
plt.ylabel("Ergotropy lower bound")

# ---- grid ----
ax.grid(True, which='major', linestyle='--', linewidth=0.5)
ax.grid(True, which='minor', linestyle=':', linewidth=0.2)

# ticks small
for lab in (ax.get_xticklabels() + ax.get_yticklabels()):
    lab.set_fontsize(9)

ax.legend(fontsize=8, loc="lower right")
plt.tight_layout()

fname = f"ergotropy_vs_Ns_superposition_multishots_ANNNI_L{L}_no2000shots.pdf"
plt.savefig(fname, dpi=300, bbox_inches='tight')
print(f"Saved figure: {fname}")

plt.show()


Saved figure: ergotropy_vs_Ns_superposition_multishots_ANNNI_L5_no2000shots.pdf


/var/folders/bl/2qzs5zlx0_b0dtwkqzfj3fznfc_zz4/T/ipykernel_2534/2463168440.py:114: UserWarning: FigureCanvasPS is non-interactive, and thus cannot be shown
  plt.show()
